# Day 37: Build a KV Cache Manager

**Layer:** Implementation | **Prerequisite:** Previous days


## Concept Overview

Implements a block allocator for KV cache management: fixed-size blocks, O(1) allocation and deallocation, LRU eviction under memory pressure, and sequence extension as new tokens are generated. This is the core of vLLM's PagedAttention implementation.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. KV Cache Block Allocator


In [ ]:
from collections import OrderedDict
import heapq

class Block:
    def __init__(self, block_id, size):
        self.id = block_id
        self.size = size
        self.ref_count = 0
        self.last_used = 0

class KVCacheManager:
    def __init__(self, total_blocks, block_size):
        self.total_blocks = total_blocks
        self.block_size = block_size
        self.blocks = {i: Block(i, block_size) for i in range(total_blocks)}
        self.free = set(range(total_blocks))
        self.allocated = {}  # seq_id -> [block_ids]
        self.clock = 0

    def allocate(self, seq_id, n_tokens):
        """Allocate blocks for a sequence."""
        n_blocks = (n_tokens + self.block_size - 1) // self.block_size
        if n_blocks > len(self.free):
            return None  # OOM
        block_ids = [self.free.pop() for _ in range(n_blocks)]
        for bid in block_ids:
            self.blocks[bid].ref_count = 1
            self.blocks[bid].last_used = self.clock
        self.allocated[seq_id] = block_ids
        self.clock += 1
        return block_ids

    def extend(self, seq_id):
        """Add one more block for a growing sequence."""
        if not self.free:
            evicted = self._evict_lru()
            if evicted is None:
                return False
        bid = self.free.pop()
        self.blocks[bid].ref_count = 1
        self.blocks[bid].last_used = self.clock
        self.allocated[seq_id].append(bid)
        self.clock += 1
        return True

    def free_seq(self, seq_id):
        if seq_id in self.allocated:
            for bid in self.allocated.pop(seq_id):
                self.blocks[bid].ref_count = 0
                self.free.add(bid)

    def _evict_lru(self):
        """Evict least recently used block (among those with ref_count==1)."""
        candidates = [(self.blocks[bid].last_used, bid)
                     for bid in range(self.total_blocks)
                     if self.blocks[bid].ref_count == 1 and
                        bid not in self.free]
        if not candidates:
            return None
        _, bid = min(candidates)
        # Find which sequence owns this block
        for seq_id, blocks in self.allocated.items():
            if bid in blocks:
                blocks.remove(bid)
                break
        self.free.add(bid)
        return bid

    @property
    def utilization(self):
        return 1 - len(self.free) / self.total_blocks

# Simulate a serving session
mgr = KVCacheManager(total_blocks=32, block_size=16)
print('KV Cache Manager simulation:')
print(f'Total capacity: {mgr.total_blocks * mgr.block_size} tokens')
events = [
    ('alloc', 'req1', 48),  ('alloc', 'req2', 80),
    ('alloc', 'req3', 32),  ('extend', 'req1', None),
    ('free', 'req2', None), ('alloc', 'req4', 96),
]
for op, seq_id, n in events:
    if op == 'alloc':
        result = mgr.allocate(seq_id, n)
        print(f'  alloc {seq_id} ({n} tok): blocks={result}, util={mgr.utilization:.0%}')
    elif op == 'extend':
        ok = mgr.extend(seq_id)
        print(f'  extend {seq_id}: ok={ok}, util={mgr.utilization:.0%}')
    elif op == 'free':
        mgr.free_seq(seq_id)
        print(f'  free {seq_id}: util={mgr.utilization:.0%}')


## 2. Eviction Policy Comparison


In [ ]:
# Simulate LRU vs FIFO eviction under memory pressure
import random
random.seed(42)
np.random.seed(42)

def simulate_eviction(policy, n_requests=200, block_capacity=20):
    active_seqs = {}
    free_blocks = block_capacity
    evictions = 0
    completions = 0

    for t in range(n_requests):
        # Randomly complete a request
        if active_seqs and random.random() < 0.3:
            seq_to_free = random.choice(list(active_seqs.keys()))
            free_blocks += active_seqs.pop(seq_to_free)['blocks']
            completions += 1

        # New request arrives
        needed = random.randint(1, 5)
        if free_blocks >= needed:
            seq_id = f's{t}'
            active_seqs[seq_id] = {'blocks': needed, 'arrival': t}
            free_blocks -= needed
        else:
            # Need to evict
            if policy == 'lru':
                victim = min(active_seqs, key=lambda s: active_seqs[s]['arrival'])
            else:  # fifo
                victim = min(active_seqs, key=lambda s: active_seqs[s]['arrival'])
            free_blocks += active_seqs.pop(victim)['blocks']
            evictions += 1
            seq_id = f's{t}'
            active_seqs[seq_id] = {'blocks': needed, 'arrival': t}
            free_blocks -= needed

    return evictions, completions

for policy in ['lru', 'fifo']:
    evictions, completions = simulate_eviction(policy)
    print(f'{policy.upper()}: {evictions} evictions, {completions} completions')


## Experiments

1. Implement prefix sharing: multiple sequences with the same prefix share the same physical blocks.
2. Measure fragmentation: plot the distribution of block utilization (tokens used / block_size) at steady state.
3. Implement a priority eviction policy: evict sequences with the most remaining decode budget first.


## Key Takeaways

- See concept overview above for the key points.
- Day 37 complete.
